In [4]:
from __future__ import annotations
from typing import Tuple

from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from qiskit_nature.second_q.operators import FermionicOp


def _xyz_to_atom_string(xyz_path: str) -> str:
    """Parse an .xyz file and build the atom string for PySCF."""
    with open(xyz_path, "r") as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]

    # first line: number of atoms (not used except for sanity check)
    try:
        natoms = int(lines[0].split()[0])
    except (ValueError, IndexError):
        raise ValueError(f"First line of {xyz_path} must contain the number of atoms.")

    # second line is a comment, actual atoms start at line 3
    atom_lines = lines[2:2 + natoms]
    if len(atom_lines) != natoms:
        raise ValueError(f"Expected {natoms} atom lines, got {len(atom_lines)}.")

    atoms = []
    for line in atom_lines:
        parts = line.split()
        if len(parts) != 4:
            raise ValueError(f"Invalid XYZ atom line: {line}")
        symbol, x, y, z = parts
        atoms.append(f"{symbol} {float(x)} {float(y)} {float(z)}")

    # PySCF expects e.g. "H 0.0 0.0 0.0; H 0.0 0.0 0.735"
    return "; ".join(atoms)


def build_hamiltonian_from_xyz(
    xyz_path: str,
    basis: str = "sto3g",
    charge: int = 0,
    spin: int = 0,
    unit: DistanceUnit = DistanceUnit.ANGSTROM,
) -> Tuple[ElectronicStructureProblem, ElectronicEnergy, FermionicOp]:
    """Create a Qiskit Nature electronic Hamiltonian from an XYZ geometry."""
    atom = _xyz_to_atom_string(xyz_path)

    driver = PySCFDriver(
        atom=atom,
        basis=basis,
        charge=charge,
        spin=spin,
        unit=unit,
    )

    problem: ElectronicStructureProblem = driver.run()
    hamiltonian: ElectronicEnergy = problem.hamiltonian
    second_q_op: FermionicOp = hamiltonian.second_q_op()

    return problem, hamiltonian, second_q_op

ElectronicStructureProblem: <qiskit_nature.second_q.problems.electronic_structure_problem.ElectronicStructureProblem object at 0x7fbb4553aa20>

Integral coefficients (alpha):
Polynomial Tensor
 "+-":
array([[-1.21782603e+00, -1.72936294e-16],
       [-2.22836603e-16, -5.09637874e-01]])
 "++--":
array([6.63330149e-01, 2.42861287e-16, 1.84626784e-01, 6.53441372e-01,
       1.38777878e-16, 6.86791536e-01])

Fermionic second-quantized Hamiltonian:
Fermionic Operator
number spin orbitals=4, number terms=36
  -1.2178260299951054 * ( +_0 -_0 )
+ -0.5096378744364828 * ( +_1 -_1 )
+ -1.2178260299951054 * ( +_2 -_2 )
+ -0.5096378744364828 * ( +_3 -_3 )
+ 0.33166507443180804 * ( +_0 +_0 -_0 -_0 )
+ 0.3267206861819477 * ( +_0 +_1 -_1 -_0 )
+ 0.33166507443180804 * ( +_0 +_2 -_2 -_0 )
+ 0.3267206861819477 * ( +_0 +_3 -_3 -_0 )
+ 0.09231339177803063 * ( +_0 +_0 -_1 -_1 )
+ 0.09231339177803063 * ( +_0 +_1 -_0 -_1 )
+ 0.09231339177803063 * ( +_0 +_2 -_3 -_1 )
+ 0.09231339177803063 * ( +_0 +_3 -_2 -_1 )

In [ ]:
# Example usage for ../test_molecules/h2.xyz
xyz_file = "../test_molecules/h2.xyz"
problem, hamiltonian, fermionic_hamiltonian = build_hamiltonian_from_xyz(xyz_file)

print("ElectronicStructureProblem:", problem)
print("\nIntegral coefficients (alpha):")
print(hamiltonian.electronic_integrals.alpha)
print("\nFermionic second-quantized Hamiltonian:")
print(fermionic_hamiltonian)
print("\nNuclear repulsion energy (not in fermionic operator):")
print(hamiltonian.nuclear_repulsion_energy)

In [5]:
from qiskit_nature.second_q.mappers import JordanWignerMapper

# fermionic_hamiltonian comes from build_hamiltonian_from_xyz(...)
jw_mapper = JordanWignerMapper()

qubit_hamiltonian = jw_mapper.map(fermionic_hamiltonian)

print("Qubit Hamiltonian (Jordan–Wigner):")
print(qubit_hamiltonian)

Qubit Hamiltonian (Jordan–Wigner):
SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IZII', 'ZIII', 'IIZZ', 'IZIZ', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.8288055 +0.j,  0.16251649+0.j, -0.19744294+0.j,  0.16251649+0.j,
 -0.19744294+0.j,  0.11720365+0.j,  0.16583254+0.j,  0.16336034+0.j,
  0.0461567 +0.j,  0.0461567 +0.j,  0.0461567 +0.j,  0.0461567 +0.j,
  0.16336034+0.j,  0.17169788+0.j,  0.11720365+0.j])
